## Penanaman

Dalam contoh kami sebelum ini, kami beroperasi pada vektor beg perkataan berdimensi tinggi dengan panjang `vocab_size`, dan kami secara jelas menukar daripada vektor perwakilan posisi berdimensi rendah kepada perwakilan one-hot yang jarang. Perwakilan one-hot ini tidak cekap dari segi memori, selain itu, setiap perkataan dianggap secara bebas antara satu sama lain, iaitu, vektor yang disandi one-hot tidak menyatakan sebarang persamaan semantik antara perkataan.

Dalam unit ini, kami akan terus meneroka dataset **News AG**. Untuk memulakan, mari muatkan data dan dapatkan beberapa definisi dari buku nota sebelum ini.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Apakah itu embedding?

Idea **embedding** adalah untuk mewakili perkataan dengan vektor padat berdimensi rendah, yang dengan cara tertentu mencerminkan makna semantik perkataan tersebut. Kita akan membincangkan kemudian bagaimana untuk membina word embedding yang bermakna, tetapi buat masa ini mari kita anggap embedding sebagai cara untuk mengurangkan dimensi vektor perkataan.

Jadi, lapisan embedding akan mengambil perkataan sebagai input, dan menghasilkan vektor output dengan dimensi yang ditetapkan `embedding_size`. Dalam erti kata tertentu, ia sangat mirip dengan lapisan `Linear`, tetapi bukannya mengambil vektor one-hot encoded, ia akan dapat mengambil nombor perkataan sebagai input.

Dengan menggunakan lapisan embedding sebagai lapisan pertama dalam rangkaian kami, kita boleh beralih dari bag-of-words kepada model **embedding bag**, di mana kita pertama menukar setiap perkataan dalam teks kita kepada embedding yang sepadan, dan kemudian mengira beberapa fungsi agregat ke atas semua embedding itu, seperti `sum`, `average` atau `max`.

![Imej yang menunjukkan pengelasan embedding untuk lima perkataan urutan.](../../../../../translated_images/ms/embedding-classifier-example.b77f021a7ee67eee.webp)

Rangkaian neural pengelasan kita akan bermula dengan lapisan embedding, kemudian lapisan agregasi, dan pengelasan linear di atasnya:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Menangani saiz urutan yang berubah-ubah

Sebagai hasil daripada seni bina ini, minibatch kepada rangkaian kami perlu dicipta dengan cara tertentu. Dalam unit sebelumnya, apabila menggunakan bag-of-words, semua tensor BoW dalam minibatch mempunyai saiz yang sama `vocab_size`, tanpa mengira panjang sebenar urutan teks kami. Apabila kami beralih ke embedding kata, kami akan berakhir dengan bilangan perkataan yang berubah-ubah dalam setiap sampel teks, dan apabila menggabungkan sampel-sampel tersebut ke dalam minibatch, kami perlu menggunakan sedikit padding.

Ini boleh dilakukan menggunakan teknik yang sama dengan memberikan fungsi `collate_fn` kepada sumber data:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Melatih pengelas embedding

Sekarang bahawa kita telah menentukan pemuat data yang sesuai, kita boleh melatih model menggunakan fungsi latihan yang telah kita tentukan dalam unit sebelumnya:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Nota**: Kami hanya melatih untuk 25k rekod di sini (kurang dari satu epok penuh) demi menjimatkan masa, tetapi anda boleh terus melatih, menulis fungsi untuk melatih selama beberapa epok, dan mencuba parameter kadar pembelajaran untuk mencapai ketepatan yang lebih tinggi. Anda sepatutnya boleh mencapai ketepatan kira-kira 90%.


### Lapisan EmbeddingBag dan Perwakilan Urutan Panjang Berubah

Dalam seni bina sebelumnya, kami perlu mengisi semua urutan kepada panjang yang sama untuk memuatkan mereka ke dalam minibatch. Ini bukan cara yang paling cekap untuk mewakili urutan panjang berubah - satu pendekatan lain adalah menggunakan vektor **offset**, yang memegang offset semua urutan yang disimpan dalam satu vektor besar.

![Imej yang menunjukkan perwakilan urutan offset](../../../../../translated_images/ms/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Nota**: Pada gambar di atas, kami menunjukkan satu urutan aksara, tetapi dalam contoh kami kami bekerja dengan urutan perkataan. Namun, prinsip umum perwakilan urutan dengan vektor offset tetap sama.

Untuk bekerja dengan perwakilan offset, kami menggunakan lapisan [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Ia serupa dengan `Embedding`, tetapi ia mengambil vektor kandungan dan vektor offset sebagai input, dan juga termasuk lapisan purata, yang boleh menjadi `mean`, `sum` atau `max`.

Berikut adalah rangkaian yang diubah suai yang menggunakan `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Untuk menyediakan set data bagi latihan, kita perlu menyediakan fungsi penukaran yang akan menyediakan vektor offset:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Perlu diingat, bahawa tidak seperti dalam semua contoh sebelumnya, rangkaian kami kini menerima dua parameter: vektor data dan vektor offset, yang mempunyai saiz yang berbeza. Begitu juga, pemuat data kami juga menyediakan 3 nilai dan bukannya 2: kedua-dua vektor teks dan offset disediakan sebagai ciri. Oleh itu, kami perlu menyesuaikan sedikit fungsi latihan kami untuk menguruskan perkara itu:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Embedding Semantik: Word2Vec

Dalam contoh kami sebelum ini, lapisan embedding model telah belajar memetakan perkataan ke representasi vektor, namun, representasi ini tidak mempunyai banyak makna semantik. Adalah baik untuk mempelajari representasi vektor sedemikian, di mana perkataan yang serupa atau sinonim akan berkaitan dengan vektor yang hampir antara satu sama lain dari segi jarak vektor tertentu (contohnya jarak euclidean).

Untuk melakukan itu, kita perlu melatih model embedding kami terlebih dahulu menggunakan koleksi teks yang besar dalam cara tertentu. Salah satu cara pertama untuk melatih embedding semantik dipanggil [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Ia berdasarkan dua seni bina utama yang digunakan untuk menghasilkan representasi terdistribusi bagi perkataan:

 - **Continuous bag-of-words** (CBoW) — dalam seni bina ini, kami melatih model untuk meramalkan perkataan dari konteks sekitar. Diberi ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, tujuan model adalah untuk meramalkan $W_0$ daripada $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** adalah sebaliknya daripada CBoW. Model menggunakan tingkap konteks sekitar perkataan untuk meramalkan perkataan semasa.

CBoW lebih pantas, manakala skip-gram lebih perlahan, tetapi melakukan tugas yang lebih baik dalam mewakili perkataan yang jarang berlaku.

![Imej menunjukkan kedua-dua algoritma CBoW dan Skip-Gram untuk menukar perkataan kepada vektor.](../../../../../translated_images/ms/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Untuk bereksperimen dengan word2vec embedding yang telah dilatih terlebih dahulu pada dataset Google News, kita boleh menggunakan perpustakaan **gensim**. Di bawah, kita dapati perkataan yang paling serupa dengan 'neural'

> **Nota:** Apabila anda mula-mula membuat vektor perkataan, memuat turunnya boleh mengambil sedikit masa!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Kita juga boleh mengira penyisipan vektor daripada perkataan itu, untuk digunakan dalam melatih model pengkelasan (kami hanya menunjukkan 20 komponen pertama vektor untuk kejelasan):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Perkara hebat tentang embedding semantik ialah anda boleh mengubah suai pengekodan vektor untuk menukar semantik. Sebagai contoh, kita boleh minta untuk mencari satu perkataan, yang perwakilan vektor terdekatnya dengan perkataan *king* dan *woman*, dan sejauh mungkin dari perkataan *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Kedua-dua CBoW dan Skip-Grams adalah pengekodan “ramalan”, kerana ia hanya mengambil kira konteks tempatan. Word2Vec tidak memanfaatkan konteks global.

**FastText**, membina atas Word2Vec dengan mempelajari representasi vektor untuk setiap perkataan dan n-gram aksara yang ditemui dalam setiap perkataan. Nilai representasi itu kemudian purata menjadi satu vektor pada setiap langkah latihan. Walaupun ini menambah banyak pengiraan tambahan untuk pra-latihan, ia membolehkan pengekodan perkataan untuk mengekod maklumat sub-perkataan.

Satu lagi kaedah, **GloVe**, menggunakan idea matriks ko-kejadian, menggunakan kaedah neural untuk menguraikan matriks ko-kejadian menjadi vektor perkataan yang lebih ekspresif dan tidak linear.

Anda boleh bermain dengan contoh ini dengan menukar pengekodan kepada FastText dan GloVe, kerana gensim menyokong beberapa model pengekodan perkataan yang berbeza.


## Menggunakan Embedding Pra-Latih dalam PyTorch

Kita boleh mengubah contoh di atas untuk mengisi matriks dalam lapisan embedding kita dengan embedding semantik, seperti Word2Vec. Kita perlu mengambil kira bahawa kosa kata embedding pra-latih dan korpus teks kita mungkin tidak sepadan, jadi kita akan menginisialisasi berat untuk perkataan yang hilang dengan nilai rawak:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Sekarang mari latih model kita. Perlu diingat bahawa masa yang diambil untuk melatih model adalah jauh lebih lama daripada contoh sebelumnya, disebabkan oleh saiz lapisan embedding yang lebih besar, dan oleh itu bilangan parameter yang jauh lebih tinggi. Juga, disebabkan oleh ini, kita mungkin perlu melatih model kita pada lebih banyak contoh jika kita ingin mengelakkan overfitting.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Dalam kes kami, kami tidak melihat peningkatan ketepatan yang besar, yang mungkin disebabkan oleh perbendaharaan kata yang berbeza. 
Untuk mengatasi masalah perbendaharaan kata yang berbeza, kami boleh menggunakan salah satu daripada penyelesaian berikut:
* Melatih semula model word2vec pada perbendaharaan kata kami
* Memuatkan set data kami dengan perbendaharaan kata dari model word2vec yang telah dilatih sebelumnya. Perbendaharaan kata yang digunakan untuk memuatkan set data boleh ditentukan semasa memuatkan.

Pendekatan yang kedua kelihatan lebih mudah, terutamanya kerana rangka kerja PyTorch `torchtext` mengandungi sokongan terbina dalam untuk embedding. Kita boleh, contohnya, memulakan perbendaharaan kata berasaskan GloVe dengan cara berikut:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Perbendaharaan kata yang dimuat mempunyai operasi asas berikut:
* `vocab.stoi` kamus membolehkan kita menukar perkataan menjadi indeks kamusnya
* `vocab.itos` melakukan sebaliknya - menukar nombor kepada perkataan
* `vocab.vectors` adalah tatasusunan vektor embedding, jadi untuk mendapatkan embedding bagi sebuah perkataan `s` kita perlu menggunakan `vocab.vectors[vocab.stoi[s]]`

Berikut adalah contoh manipulasi embedding untuk menunjukkan persamaan **kind-man+woman = queen** (Saya perlu melaraskan pekali sedikit untuk menjadikannya berfungsi):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Untuk melatih pengklasifikasi menggunakan pengekodan tersebut, kita perlu terlebih dahulu mengekodkan set data kita menggunakan perbendaharaan kata GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Seperti yang telah kita lihat di atas, semua penanaman vektor disimpan dalam matriks `vocab.vectors`. Ia memudahkan sangat untuk memuatkan berat itu ke dalam berat lapisan penanaman dengan menggunakan penyalinan mudah:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Sekarang mari kita latih model kita dan lihat jika kita mendapat keputusan yang lebih baik:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Salah satu sebab kita tidak melihat peningkatan ketepatan yang ketara adalah kerana beberapa perkataan daripada set data kami tidak ada dalam perbendaharaan kata GloVe yang telah dilatih terlebih dahulu, dan oleh itu ia secara asasnya diabaikan. Untuk mengatasi hakikat ini, kita boleh melatih embedding kita sendiri pada set data kita. 


## Pengertian Embedding Kontekstual

Satu had utama bagi perwakilan embedding pra-latihan tradisional seperti Word2Vec ialah masalah penyingkiran makna perkataan. Walaupun embedding pra-latihan boleh menangkap sebahagian daripada makna perkataan dalam konteks, setiap maksud perkataan yang mungkin disandarkan dalam embedding yang sama. Ini boleh menyebabkan masalah dalam model hiliran, kerana banyak perkataan seperti perkataan 'main' mempunyai maksud yang berbeza bergantung pada konteks penggunaannya.

Contohnya, perkataan 'main' dalam dua ayat berbeza ini mempunyai maksud yang agak berbeza:
- Saya pergi ke sebuah **pementasan** di teater.
- John mahu **bermain** dengan kawan-kawannya.

Embedding pra-latihan di atas mewakili kedua-dua maksud perkataan 'main' ini dalam embedding yang sama. Untuk mengatasi had ini, kita perlu membina embedding berdasarkan **model bahasa**, yang dilatih pada korpus teks yang besar, dan *tahu* bagaimana perkataan boleh disusun dalam konteks yang berbeza. Perbincangan mengenai embedding kontekstual adalah di luar skop tutorial ini, tetapi kita akan kembali kepada topik ini apabila bercakap tentang model bahasa dalam unit seterusnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
